In [ ]:
///- --package ..
:< sm'_operators
:< engine

#### post (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:| block =
    | Heading : string
    | Paragraph : string
    | Code : string * string
    | Terminal : string * string
    | Table : list string * list (list string)
    | Note : string
    | Rule

:= ^ cells (tag : string) (row : list string) : string =
    ?= row ?|
    | [] => ""
    | cell :: rest => "<" ++# tag ++# ">" ++# html.escape cell ++# "</" ++# tag ++# ">" ++# cells tag rest

:= ^ body_rows (rows : list (list string)) : string =
    ?= rows ?|
    | [] => ""
    | row :: rest => "<tr>" ++# cells "td" row ++# "</tr>" ++# body_rows rest

:= render_block (b : block) : string =
    ?= b ?|
    | Heading text => "<h2>" ++# html.escape text ++# "</h2>"
    | Paragraph text => "<p>" ++# html.escape text ++# "</p>"
    | Code (language, text) =>
        := opening = "<pre><code class=\"language-" ++# html.escape language
        := closing = "\">"
        (opening ++# closing) ++# (html.escape text ++# "</code></pre>")
    | Terminal (command, output) =>
        "<pre class=\"terminal\"><code><span class=\"prompt\">$ </span>"
        ++# html.escape command
        ++# "\n"
        ++# html.escape output
        ++# "</code></pre>"
    | Table (head, rows) =>
        "<table><thead><tr>" ++# cells "th" head ++# "</tr></thead><tbody>" ++# body_rows rows ++# "</tbody></table>"
    | Note text => "<aside class=\"note\">" ++# html.escape text ++# "</aside>"
    | Rule => "<hr class=\"section-rule\">"

#### a_heading_is_escaped (test)

In [1]:
///- --test --symbol
///> zig

render_block (Heading "Async & DNS") |> _assert_eq "<h2>Async &amp; DNS</h2>"

__assert_eq { expected = <h2>Async &amp; DNS</h2> }

#### code_keeps_its_language (test)

In [2]:
///- --test --symbol
///> zig

render_block (Code ("spiral", "inl x = 1 < 2"))
|> _assert_eq "<pre><code class=\"language-spiral\">inl x = 1 &lt; 2</code></pre>"

__assert_eq { expected = <pre><code class="language-spiral">inl x = 1 &lt; 2</code></pre> }

#### a_rule_separates_sections (test)

In [3]:
///- --test --symbol
///> zig

render_block Rule |> _assert_eq "<hr class=\"section-rule\">"

__assert_eq { expected = <hr class="section-rule"> }

#### a_terminal_block_shows_the_command_and_its_output (test)

In [4]:
///- --test --symbol
///> zig

render_block (Terminal ("hyperfine 'spiral build'", "Time (mean): 1.2 s"))
|> _assert_eq (
    "<pre class=\"terminal\"><code><span class=\"prompt\">$ </span>hyperfine 'spiral build'\n"
    ^^ "Time (mean): 1.2 s</code></pre>"
)

__assert_eq { expected = <pre class="terminal"><code><span class="prompt">$ </span>hyperfine 'spiral build'
Time (mean): 1.2 s</code></pre> }

#### a_table_has_a_head_and_escaped_cells (test)

In [5]:
///- --test --symbol
///> zig

render_block (Table ([ "Backend"; "Rows" ], [ [ "C"; "< 500" ] ]))
|> _assert_eq (
    "<table><thead><tr><th>Backend</th><th>Rows</th></tr></thead><tbody><tr><td>C</td><td>&lt; "
    ^^ "500</td></tr></tbody></table>"
)

__assert_eq { expected = <table><thead><tr><th>Backend</th><th>Rows</th></tr></thead><tbody><tr><td>C</td><td>&lt; 500</td></tr></tbody></table> }

#### a_note_is_an_aside (test)

In [6]:
///- --test --symbol
///> zig

render_block (Note "Numbers are from one laptop.")
|> _assert_eq "<aside class=\"note\">Numbers are from one laptop.</aside>"

__assert_eq { expected = <aside class="note">Numbers are from one laptop.</aside> }